# Indic-Translate vs IndicTrans2 - accuracy & speed benchmark

**Why Colab, not local:** `bodhan-ai/indic-translate` is an 8B-parameter Gemma-4-based LLM, 15.9GB in
bf16, with no CPU or quantized inference path documented on its model card (minimum 24GB GPU
recommended). This dev machine has **no GPU and only ~16GB total RAM** - not enough to even load the
raw bf16 weights, let alone run inference. Production's `IndicTrans2` is a completely different kind of
model (dedicated ~211M-per-direction encoder-decoder NMT, CTranslate2 int8, purpose-built to run on CPU),
so this comparison is inherently GPU-LLM vs CPU-NMT, not apples-to-apples hardware - speed numbers below
should be read with that in mind.

**Before running:** `Runtime > Change runtime type` -> GPU (T4 is fine, it's the free tier default). Even
on a T4 (15GB VRAM) the raw bf16 weights barely fit with no room for KV cache, so this notebook loads the
model **8-bit quantized** (`bitsandbytes`, `load_in_8bit=True`) to leave headroom - note this is a real
deviation from the model card's own bf16-only benchmarks, so its self-reported chrF++/latency numbers
won't exactly reproduce here. That's expected and will be called out in the final summary.

**You will also need:** a Hugging Face account that has accepted the "Indic Open Model License v1.0" /
contact-info form on https://huggingface.co/bodhan-ai/indic-translate (gated repo), and a HF access token
(https://huggingface.co/settings/tokens) to paste into the login cell below.

**Test set:** the same 50-case `TEST_SET` (both en->pa and pa->en, mix of glossary-style short phrases and
general sentences) from `evaluate_translation_accuracy.py`, scored with chrF (`sacrebleu`) exactly like our
existing production benchmarks, so the final number is directly comparable to our recorded baselines:
raw IndicTrans2 (no glossary) ~78.1/100 chrF on this project's test sets, full production pipeline (with
glossary overrides) ~89.5-100/100.


In [ ]:
!nvidia-smi

## 1. Install dependencies

Needs `transformers>=5.12` (ships the Gemma 4 architecture) and `torch>=2.6`. `bitsandbytes` is for
8-bit loading, `sacrebleu` is for chrF scoring (same metric/library used in `evaluate_translation_accuracy.py`).


In [ ]:
!pip install --quiet --upgrade "transformers>=5.12" "torch>=2.6" accelerate bitsandbytes sacrebleu huggingface_hub

## 2. Log in to Hugging Face (required for the gated `bodhan-ai/indic-translate` repo)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load Indic-Translate (8-bit quantized, to fit a free-tier T4)

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL = "bodhan-ai/indic-translate"

quant_config = BitsAndBytesConfig(load_in_8bit=True)

processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL,
    quantization_config=quant_config,
    device_map="auto",
    attn_implementation="sdpa",
).eval()

print("Model loaded. Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else model.device)

## 4. Test set

Same 50-case `TEST_SET` as `evaluate_translation_accuracy.py` (copied here so this notebook is
self-contained on Colab - no need to upload any project files).

In [ ]:
# (source_lang, target_lang, input_text, reference_translation_or_list_of_acceptable_translations)
# Copied verbatim from evaluate_translation_accuracy.py's TEST_SET.
TEST_SET = [
    ("en", "pa", "Hello", "ਸਤ ਸ੍ਰੀ ਅਕਾਲ"),
    ("en", "pa", "Thank you very much", "ਬਹੁਤ ਧੰਨਵਾਦ"),
    ("pa", "en", "ਬਾਥਰੂਮ ਕਿੱਥੇ ਹੈ?", "Where is the bathroom?"),
    ("pa", "en", "ਮੈਨੂੰ ਸਮਝ ਨਹੀਂ ਆਈ", "I don't understand"),
    ("en", "pa", "I am going to the market to buy vegetables.",
     "ਮੈਂ ਸਬਜ਼ੀਆਂ ਖਰੀਦਣ ਲਈ ਬਾਜ਼ਾਰ ਜਾ ਰਿਹਾ ਹਾਂ।"),
    ("en", "pa", "The weather is very nice today, so let's go for a walk.",
     ["ਅੱਜ ਮੌਸਮ ਬਹੁਤ ਵਧੀਆ ਹੈ, ਇਸ ਲਈ ਚਲੋ ਸੈਰ ਲਈ ਚੱਲੀਏ।",
      "ਅੱਜ ਮੌਸਮ ਬਹੁਤ ਚੰਗਾ ਹੈ, ਇਸ ਲਈ ਆਓ ਸੈਰ ਕਰਨ ਚੱਲੀਏ।"]),
    ("en", "pa", "Can you please help me find my way to the train station?",
     ["ਕੀ ਤੁਸੀਂ ਮੈਨੂੰ ਰੇਲਵੇ ਸਟੇਸ਼ਨ ਦਾ ਰਸਤਾ ਲੱਭਣ ਵਿੱਚ ਮਦਦ ਕਰ ਸਕਦੇ ਹੋ?",
      "ਕੀ ਤੁਸੀਂ ਰੇਲਵੇ ਸਟੇਸ਼ਨ ਤੱਕ ਪਹੁੰਚਣ ਦਾ ਰਸਤਾ ਲੱਭਣ ਵਿੱਚ ਮੇਰੀ ਮਦਦ ਕਰ ਸਕਦੇ ਹੋ?"]),
    ("pa", "en", "ਮੈਂ ਸਕੂਲ ਜਾ ਰਿਹਾ ਹਾਂ।", ["I am going to school.", "I'm going to school."]),
    ("pa", "en", "ਕੱਲ੍ਹ ਮੇਰੇ ਦੋਸਤ ਦਾ ਜਨਮਦਿਨ ਹੈ।", "Tomorrow is my friend's birthday."),
    ("pa", "en", "ਕਿਰਪਾ ਕਰਕੇ ਦਰਵਾਜ਼ਾ ਬੰਦ ਕਰ ਦਿਓ।", "Please close the door."),
    ("en", "pa", "Good morning", "ਸ਼ੁਭ ਸਵੇਰ"),
    ("en", "pa", "Nice to meet you", "ਤੁਹਾਨੂੰ ਮਿਲ ਕੇ ਖੁਸ਼ੀ ਹੋਈ"),
    ("en", "pa", "How much is the ticket?", "ਟਿਕਟ ਕਿੰਨੇ ਦੀ ਹੈ?"),
    ("en", "pa", "Turn left", "ਖੱਬੇ ਮੁੜੋ"),
    ("pa", "en", "ਸ਼ੁਭ ਸਵੇਰ", "Good morning"),
    ("pa", "en", "ਅਲਵਿਦਾ", "Goodbye"),
    ("pa", "en", "ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ?", "How are you?"),
    ("pa", "en", "ਹਸਪਤਾਲ ਕਿੱਥੇ ਹੈ?", "Where is the hospital?"),
    ("en", "pa", "Where is the train station?", "ਰੇਲਵੇ ਸਟੇਸ਼ਨ ਕਿੱਥੇ ਹੈ?"),
    ("en", "pa", "I need a doctor", "ਮੈਨੂੰ ਡਾਕਟਰ ਚਾਹੀਦਾ ਹੈ"),
    ("en", "pa", "Call an ambulance", "ਐਂਬੂਲੈਂਸ ਬੁਲਾਓ"),
    ("en", "pa", "This is an emergency", "ਇਹ ਇੱਕ ਐਮਰਜੈਂਸੀ ਹੈ"),
    ("pa", "en", "ਮੈਨੂੰ ਡਾਕਟਰ ਚਾਹੀਦਾ ਹੈ", "I need a doctor"),
    ("pa", "en", "ਐਂਬੂਲੈਂਸ ਬੁਲਾਓ", "Call an ambulance"),
    ("pa", "en", "ਪੁਲਿਸ ਨੂੰ ਬੁਲਾਓ", "Call the police"),
    ("pa", "en", "ਮੈਨੂੰ ਬੁਖਾਰ ਹੈ", "I have a fever"),
    ("en", "pa", "Monday", "ਸੋਮਵਾਰ"),
    ("en", "pa", "Seven", "ਸੱਤ"),
    ("en", "pa", "Water", "ਪਾਣੀ"),
    ("pa", "en", "ਸ਼ੁੱਕਰਵਾਰ", "Friday"),
    ("pa", "en", "ਦਸ", "Ten"),
    ("pa", "en", "ਖਾਣਾ", "Food"),
    ("en", "pa", "My phone battery is almost dead.",
     ["ਮੇਰੇ ਫ਼ੋਨ ਦੀ ਬੈਟਰੀ ਲਗਭਗ ਖਤਮ ਹੋ ਗਈ ਹੈ।", "ਮੇਰੇ ਫ਼ੋਨ ਦੀ ਬੈਟਰੀ ਲਗਭਗ ਖ਼ਤਮ ਹੋ ਚੁੱਕੀ ਹੈ।"]),
    ("en", "pa", "She works at a hospital in the city.",
     "ਉਹ ਸ਼ਹਿਰ ਦੇ ਇੱਕ ਹਸਪਤਾਲ ਵਿੱਚ ਕੰਮ ਕਰਦੀ ਹੈ।"),
    ("en", "pa", "We will meet at the restaurant at seven o'clock.",
     "ਅਸੀਂ ਸੱਤ ਵਜੇ ਰੈਸਟੋਰੈਂਟ ਵਿੱਚ ਮਿਲਾਂਗੇ।"),
    ("en", "pa", "Please send me the documents by email.",
     ["ਕਿਰਪਾ ਕਰਕੇ ਮੈਨੂੰ ਦਸਤਾਵੇਜ਼ ਈਮੇਲ ਰਾਹੀਂ ਭੇਜੋ।", "ਕਿਰਪਾ ਕਰਕੇ ਮੈਨੂੰ ਈਮੇਲ ਰਾਹੀਂ ਦਸਤਾਵੇਜ਼ ਭੇਜੋ।"]),
    ("en", "pa", "I would like to book a room for two nights.",
     ["ਮੈਂ ਦੋ ਰਾਤਾਂ ਲਈ ਇੱਕ ਕਮਰਾ ਬੁੱਕ ਕਰਨਾ ਚਾਹੁੰਦਾ ਹਾਂ।", "ਮੈਂ ਦੋ ਰਾਤਾਂ ਲਈ ਕਮਰਾ ਬੁੱਕ ਕਰਨਾ ਚਾਹੁੰਦਾ ਹਾਂ।"]),
    ("en", "pa", "The train is running thirty minutes late.",
     "ਰੇਲਗੱਡੀ ਤੀਹ ਮਿੰਟ ਦੇਰੀ ਨਾਲ ਚੱਲ ਰਹੀ ਹੈ।"),
    ("en", "pa", "Can I have the bill, please?", "ਕੀ ਮੈਨੂੰ ਬਿੱਲ ਮਿਲ ਸਕਦਾ ਹੈ?"),
    ("en", "pa", "I lost my passport at the airport.",
     ["ਮੈਂ ਹਵਾਈ ਅੱਡੇ 'ਤੇ ਆਪਣਾ ਪਾਸਪੋਰਟ ਗੁਆ ਦਿੱਤਾ।", "ਮੇਰਾ ਪਾਸਪੋਰਟ ਹਵਾਈ ਅੱਡੇ ਉੱਤੇ ਗੁੰਮ ਹੋ ਗਿਆ।"]),
    ("en", "pa", "Please keep the change.", "ਕਿਰਪਾ ਕਰਕੇ ਬਾਕੀ ਪੈਸੇ ਰੱਖੋ।"),
    ("pa", "en", "ਮੈਂ ਹਰ ਰੋਜ਼ ਸਵੇਰੇ ਸੈਰ ਕਰਨ ਜਾਂਦਾ ਹਾਂ।", "I go for a walk every morning."),
    ("pa", "en", "ਉਸਨੇ ਬਹੁਤ ਮਿਹਨਤ ਨਾਲ ਪ੍ਰੀਖਿਆ ਪਾਸ ਕੀਤੀ।",
     ["He passed the exam with great effort.", "He passed the exam with a lot of hard work."]),
    ("pa", "en", "ਕੀ ਤੁਸੀਂ ਮੈਨੂੰ ਇੱਕ ਗਲਾਸ ਪਾਣੀ ਦੇ ਸਕਦੇ ਹੋ?", "Can you give me a glass of water?"),
    ("pa", "en", "ਬੱਚੇ ਬਾਗ਼ ਵਿੱਚ ਖੇਡ ਰਹੇ ਹਨ।", "The children are playing in the garden."),
    ("pa", "en", "ਮੌਸਮ ਵਿਭਾਗ ਨੇ ਕੱਲ੍ਹ ਮੀਂਹ ਦੀ ਭਵਿੱਖਬਾਣੀ ਕੀਤੀ ਹੈ।",
     ["The weather department has forecast rain for tomorrow.",
      "The Meteorological Department has predicted rain tomorrow."]),
    ("pa", "en", "ਮੈਨੂੰ ਇਹ ਕਿਤਾਬ ਬਹੁਤ ਪਸੰਦ ਆਈ।",
     ["I really liked this book.", "Absolutely loved this book."]),
    ("pa", "en", "ਦੁਕਾਨ ਸ਼ਾਮ ਨੂੰ ਅੱਠ ਵਜੇ ਬੰਦ ਹੋ ਜਾਂਦੀ ਹੈ।",
     ["The shop closes at eight in the evening.", "The shop closes at eight o'clock in the evening."]),
    ("pa", "en", "ਮੈਨੂੰ ਇੱਕ ਟੈਕਸੀ ਦੀ ਲੋੜ ਹੈ।", "I need a taxi."),
    ("pa", "en", "ਸਟੇਸ਼ਨ ਤੋਂ ਬਾਜ਼ਾਰ ਕਿੰਨਾ ਦੂਰ ਹੈ?", "How far is the market from the station?"),
]

# Indic-Translate's prompt format needs the target language's full name, not the ISO code.
LANG_NAME = {"en": "English", "pa": "Punjabi"}

print(f"Loaded {len(TEST_SET)} test cases.")

## 5. Translate function

Follows the model card's prompting guide exactly: single `user` message, no `system` message, target
language specified by name only (never mention the source language), greedy decoding.

In [ ]:
import time


def indic_translate(text: str, target_language: str, max_new_tokens: int = 128) -> tuple[str, float]:
    messages = [{
        "role": "user",
        "content": f"Translate the following text into {target_language}:\n\n{text}",
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy, recommended for translation
            use_cache=True,
        )
    elapsed = time.perf_counter() - start

    completion = output[0, inputs["input_ids"].shape[-1]:]
    text_out = processor.decode(completion, skip_special_tokens=True).strip()
    return text_out, elapsed


# Quick smoke test before running the full 50-case set.
sample_out, sample_latency = indic_translate("The committee approved the proposal after a long debate.", "Hindi")
print(f"{sample_latency:.2f}s ->", sample_out)

## 6. Run the full test set

This is the slow cell - 50 sequential generate() calls. Progress prints after every case so you can see
it's making progress, not hung.

In [ ]:
results = []  # list of dicts: source_lang, target_lang, text, references, output, latency

for i, (source_lang, target_lang, text, reference) in enumerate(TEST_SET, start=1):
    references = [reference] if isinstance(reference, str) else reference
    output, latency = indic_translate(text, LANG_NAME[target_lang])
    results.append({
        "source_lang": source_lang,
        "target_lang": target_lang,
        "text": text,
        "references": references,
        "output": output,
        "latency": latency,
    })
    print(f"[{i}/{len(TEST_SET)}] ({source_lang}->{target_lang}, {latency:.2f}s) {text!r} -> {output!r}")

## 7. Score with chrF (same metric as `evaluate_translation_accuracy.py`)

For cases with multiple acceptable references, score is the max chrF across the reference list -
identical logic to the production benchmark script.

In [ ]:
from sacrebleu.metrics import CHRF

chrf = CHRF()

for r in results:
    r["chrf"] = max(chrf.sentence_score(r["output"], [ref]).score for ref in r["references"])

avg_chrf = sum(r["chrf"] for r in results) / len(results)
avg_latency = sum(r["latency"] for r in results) / len(results)

# Split out en->pa vs pa->en, since the model card itself reports these separately.
en_pa = [r for r in results if r["target_lang"] == "pa"]
pa_en = [r for r in results if r["target_lang"] == "en"]
avg_chrf_en_pa = sum(r["chrf"] for r in en_pa) / len(en_pa)
avg_chrf_pa_en = sum(r["chrf"] for r in pa_en) / len(pa_en)

print(f"Overall avg chrF:        {avg_chrf:.1f}/100  ({len(results)} cases)")
print(f"  en->pa avg chrF:       {avg_chrf_en_pa:.1f}/100  ({len(en_pa)} cases)")
print(f"  pa->en avg chrF:       {avg_chrf_pa_en:.1f}/100  ({len(pa_en)} cases)")
print(f"Average latency/call:    {avg_latency:.2f}s  (8-bit quantized, {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print()
print("Lowest-scoring cases (worth manually inspecting):")
for r in sorted(results, key=lambda r: r["chrf"])[:8]:
    print(f"  chrF={r['chrf']:.1f}  {r['source_lang']}->{r['target_lang']}  {r['text']!r}")
    print(f"    expected: {r['references']!r}")
    print(f"    got:      {r['output']!r}")

## 8. Compare against our recorded production baselines

- Raw IndicTrans2 (no glossary overrides), same style of test set: **~78.1/100 chrF** (`evaluate_raw_model_only.py`).
- Full production pipeline (IndicTrans2 + glossary overrides, verified via `verify=True`): **~89.5-100/100 chrF**.
- Production CT2 int8 translate latency (CPU, `performance-4x` Fly machine, `num_beams=4`): **~0.85-1.5s/request**.
- Indic-Translate's own self-reported IN22-Gen chrF++ for Punjabi (bf16, unquantized, GPU): en->pa 48.4, pa->en 59.2.
  Expect this notebook's numbers to land somewhat below the model card's own (8-bit quantization +
  different, project-specific test set, not the academic IN22-Gen split).

**When you're done:** copy the final cell's printed summary (overall/en->pa/pa->en chrF, avg latency) and a
few of the worst-scoring examples back into the chat, so the verdict can be written up and documented in
`/memories/repo/EnglishPunjabiVoiceTranslationApp.md`, matching how the IndicConformer-vs-Indic-Transcribe and
IndicTrans2-vs-NLLB-200 investigations were recorded. Nothing in production should change based on this
notebook alone - it's a research comparison, not a migration.